In [1]:
%matplotlib qt
import torch
from d2l import torch as d2l
from torch import nn as nn
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def cal(l,varphi_ofoot):
    l1 = 0.48
    l2 = 0.36
    l0 = 0.09
    c1 = 0.2
    c2 = 0.04
    c = c1+c2
    psi_02 = np.arccos((l1+l2+l0) / np.sqrt((l1+l2+l0)**2 + c2**2))
    print('psi_02:{},  psi_02<<pi/2'.format(psi_02))
    b1 = l0 * np.cos(varphi_ofoot) + c1 * np.sin(varphi_ofoot)
    print('b1:{},  b1>0'.format(b1))
    b2 = l/2 - c + c1 * np.cos(varphi_ofoot) - l0 * np.sin(varphi_ofoot)
    print('b2:{},  b2>>b1'.format(b2))
    psi_01 = np.arcsin(b1 / np.sqrt(b1**2 + b2**2))
    print('psi_01:{},  range(0,pi/2)'.format(psi_01))
    psi_max = -np.arcsin(((l1+l2+l0)**2 + c2**2 - (l1+l2)**2 - b1**2 - b2**2) / (2 * (l1+l2) * np.sqrt(b1**2 + b2**2))) + psi_01
    psi_min = -np.arccos(((l1+l2) * np.cos(psi_max) + l0 * np.cos(varphi_ofoot) + c1 * np.sin(varphi_ofoot)) / (np.sqrt((l1+l2+l0)**2 + c2**2))) - psi_02
    print('psi_hip_fore = psi_min:{},   range(-pi/2, {})'.format(psi_min,-2*psi_02))
    print('psi_hip_rear = psi_max:{}'.format(psi_max))
    
    psi_hip_fore = psi_min
    psi_hip_rear = psi_max
    return psi_hip_fore, psi_hip_rear

In [ ]:
l = np.linspace(0.4, 1.6, 1000)
l[-1] = 1.6

In [2]:
#计算函数
def cal(l,varphi_ofoot):
    #定义初始长度(:mm)
    l1, l2, lf, c1, c2 = 0.48, 0.36, 0.09, 0.2, 0.04
    l1 = torch.tensor(l1)
    cf = c1 + c2
    #计算角度psi02
    psi02 = torch.acos((l1+l2+lf)/(torch.sqrt((l1+l2+lf)**2 + c2**2)))
    #定义脚夹角(:度)
    #varphi_ofoot = torch.tensor(varphi_ofoot/180*torch.pi)#弧度制转换
    ##计算b1,b2
    b1 = lf*torch.cos(varphi_ofoot) + c1*torch.sin(varphi_ofoot)
    b2 = l/2 - cf + c1 * torch.cos(varphi_ofoot) - lf * torch.sin(varphi_ofoot)
    #计算psi0
    psi0 = torch.asin(b1 / torch.sqrt(b1**2 + b2**2))
    psi_hipmax = -torch.asin(((l1+l2+lf)**2 + c2**2 - (l1+l2)**2 - b1**2 - b2**2) /(2*(l1+l2)* torch.sqrt(b1**2 + b2**2))) + psi0
    psi_hipmin = -torch.arccos(((l1+l2)*torch.cos(psi_hipmax) + lf*torch.cos(varphi_ofoot) + c1*torch.sin(varphi_ofoot))/(torch.sqrt((l1+l2+lf)**2 + c2**2))) - psi02
    return psi_hipmax/torch.pi*180, psi_hipmin/torch.pi*180

In [55]:
import matplotlib.pyplot as plt
import numpy as np



# Constants from the paper
l1 = 0.48  # thigh length (m)
l2 = 0.36  # shank length (m)
l0 = 0.09  # foot height from ankle joint (m)
c1 = 0.2   # foot arch length (m)
c2 = 0.04  # heel length (m)
psi_0_double_prime = 0.043  # constant angle (rad)
c = c1 + c2  # total foot length (m)

# Stride length range: l >= 4*c2 = 0.16 m, up to a reasonable maximum (e.g., 1.5 m)
l = np.linspace(0.7, 1.6,1000)
l[-1] = 1.6

# Foot-ground angles to plot
phi_foot_0_values = [np.pi/32, np.pi/18, np.pi/12]  # radians

# Initialize plot
plt.figure(figsize=(7,6),dpi=300)
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

for phi_foot_0 in phi_foot_0_values:
    # Compute b1 and b2
    b1 = l0 * np.cos(phi_foot_0) + c1 * np.sin(phi_foot_0)
    b2 = l / 2 - c + c1 * np.cos(phi_foot_0) - l0 * np.sin(phi_foot_0)

    # Compute psi_hip_max (psi_hip,flex^rear)
    term2 = (l1 + l2 + l0)**2 + c2**2 - (l1 + l2)**2 - b1**2 - b2**2
    denom2 = 2 * (l1 + l2) * np.sqrt(b1**2 + b2**2)
    psi_prime = np.arcsin(b1 / np.sqrt(b1**2 + b2**2))
    psi_hip_max = -np.arcsin(term2 / denom2) + psi_prime
    
    # Compute psi_hip_min (psi_hip,flex^fore)
    term1 = (l1 + l2) * np.cos(psi_hip_max) + l0 * np.cos(phi_foot_0) + c1 * np.sin(phi_foot_0)
    denom = np.sqrt((l1 + l2 + l0)**2 + c2**2)
    psi_0_double_prime = np.arccos((l1 + l2 + l0)/(np.sqrt((l1 +l2 + l0)**2+c2**2)))
    psi_hip_min = -np.arccos(term1 / denom) - psi_0_double_prime

    if phi_foot_0 == np.pi/32:
        plt.plot(l, psi_hip_min, label=f'$\psi_\mathrm{{hip,flex}}^\mathrm{{fore}}$, $\phi_\mathrm{{foot}}^0=\pi/36$', color='C0', linewidth=2, linestyle='-')
        plt.plot(l, psi_hip_max, label=f'$\psi_\mathrm{{hip,flex}}^\mathrm{{rear}}$, $\phi_\mathrm{{foot}}^0=\pi/36$',color='C0',linewidth=2,linestyle='--')

    if phi_foot_0 == np.pi/18:
        plt.plot(l, psi_hip_min, label=f'$\psi_\mathrm{{hip,flex}}^\mathrm{{fore}}$, $\phi_\mathrm{{foot}}^0=\pi/18$',color='C1',linewidth=2,linestyle='-')
        plt.plot(l, psi_hip_max, label=f'$\psi_\mathrm{{hip,flex}}^\mathrm{{rear}}$, $\phi_\mathrm{{foot}}^0=\pi/18$',color='C1',linewidth=2,linestyle='--')
    if phi_foot_0 == np.pi/12:
        plt.plot(l, psi_hip_min, label=f'$\psi_\mathrm{{hip,flex}}^\mathrm{{fore}}$, $\phi_\mathrm{{foot}}^0=\pi/12$',color='C2',linewidth=2,linestyle='-')
        plt.plot(l, psi_hip_max, label=f'$\psi_\mathrm{{hip,flex}}^\mathrm{{rear}}$, $\phi_\mathrm{{foot}}^0=\pi/12$',color='C2',linewidth=2,linestyle='--')

    # Plot
    

#网格

    

#网格
plt.grid()

x_ticks = np.array([0.8, 0.9, 1, 1.1, 1.2, 1.3,1.4,1.5,1.6])
x_tick_labels = ['0.8', '0.9', '1', '1.1', '1.2', '1.3','1.4','1.5','1.6']
plt.xticks(x_ticks, x_tick_labels,fontsize=20)

y_ticks = np.arange(-np.pi/6, np.pi/6+0.0001, np.pi/12)
y_tick_labels = [r'$-\frac{\pi}{6}$',r'$-\frac{\pi}{12}$',
                 '0', r'$\frac{\pi}{12}$',r'$\frac{\pi}{6}$']
plt.yticks(y_ticks, y_tick_labels,fontsize=30)
#plt.axis('equal')
#plt.legend(prop={'size':20},handlelength=1,borderpad=0.2, labelspacing=1)
plt.ylim(-np.pi/6, np.pi/6)
plt.xlim(0.8,1.6)
plt.tick_params(pad=10)

In [31]:
l = 1.6

varphi_ofoot = np.pi / 12

l1 = 0.48
l2 = 0.36
l0 = 0.09
c1 = 0.2
c2 = 0.04
c = c1+c2
psi_02 = np.arccos((l1+l2+l0) / np.sqrt((l1+l2+l0)**2 + c2**2))
print('psi_02:{},  psi_02<<pi/2'.format(psi_02))
b1 = l0 * np.cos(varphi_ofoot) + c1 * np.sin(varphi_ofoot)
print('b1:{},  b1>0'.format(b1))
b2 = l/2 - c + c1 * np.cos(varphi_ofoot) - l0 * np.sin(varphi_ofoot)
print('b2:{},  b2>>b1'.format(b2))
psi_01 = np.arcsin(b1 / np.sqrt(b1**2 + b2**2))
print('psi_01:{},  range(0,pi/2)'.format(psi_01))
psi_max = -np.arcsin(((l1+l2+l0)**2 + c2**2 - (l1+l2)**2 - b1**2 - b2**2) / (2 * (l1+l2) * np.sqrt(b1**2 + b2**2))) + psi_01
psi_min = -np.arccos(((l1+l2) * np.cos(psi_max) + l0 * np.cos(varphi_ofoot) + c1 * np.sin(varphi_ofoot)) / (np.sqrt((l1+l2+l0)**2 + c2**2))) - psi_02
print('psi_hip_fore = psi_min:{},   range(-pi/2, {})'.format(psi_min,-2*psi_02))
print('psi_hip_rear = psi_max:{}'.format(psi_max))

psi_hip_fore = psi_min
psi_hip_rear = psi_max

psi_02:0.042984259867771095,  psi_02<<pi/2
b1:0.1386971333865203,  b1>0
b2:0.7298914511985869,  b2>>b1
psi_01:0.18778542790449376,  range(0,pi/2)
psi_hip_fore = psi_min:-0.39666726921930284,   range(-pi/2, -0.08596851973554219)
psi_hip_rear = psi_max:0.5064765946248628


In [26]:
plt.plot([1,2,3],[2,3,6],label=f'$\psi_{{hip,flex}}^{{fore}}$, $\varphi_{{foot}}^0=\pi/18$')
plt.legend()
#print(f'$\varphi$')

Traceback (most recent call last):
  File "D:\miniconda\lib\site-packages\matplotlib\backends\backend_qt.py", line 468, in _draw_idle
    self.draw()
  File "D:\miniconda\lib\site-packages\matplotlib\backends\backend_agg.py", line 400, in draw
    self.figure.draw(self.renderer)
  File "D:\miniconda\lib\site-packages\matplotlib\artist.py", line 95, in draw_wrapper
    result = draw(artist, renderer, *args, **kwargs)
  File "D:\miniconda\lib\site-packages\matplotlib\artist.py", line 72, in draw_wrapper
    return draw(artist, renderer)
  File "D:\miniconda\lib\site-packages\matplotlib\figure.py", line 3175, in draw
    mimage._draw_list_compositing_images(
  File "D:\miniconda\lib\site-packages\matplotlib\image.py", line 131, in _draw_list_compositing_images
    a.draw(renderer)
  File "D:\miniconda\lib\site-packages\matplotlib\artist.py", line 72, in draw_wrapper
    return draw(artist, renderer)
  File "D:\miniconda\lib\site-packages\matplotlib\axes\_base.py", line 3064, in draw
    m

In [169]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# 参数设置（基于图4(d)的值）
ell_1 = 0.48  # 大腿长度 (m)
ell_2 = 0.36  # 小腿长度 (m)
ell_0 = 0.09  # 脚踝到足底的距离 (m)
c_1 = 0.2     # 足弓长度 (m)
c_2 = 0.04    # 脚跟长度 (m)
c = c_1 + c_2 # 脚总长度
ell = 0.4     # 步幅 (m)
phi_pitch = 0 # 骨盆俯仰角




# 定义方程组
def equations(angles, psi_knee_rear, phi_foot_rear):
    psi_hip_flex_fore, psi_hip_flex_rear = angles
    psi_knee_fore = 0  # 前膝不弯曲
    phi_foot_fore = 0  # 前脚平放在地面
    psi_ankle_fore = -psi_hip_flex_fore  # 由于 phi_foot_fore = 0
    # 计算后脚踝角
    psi_ankle_rear = phi_foot_rear - psi_hip_flex_rear - psi_knee_rear

    # 方程 (3)：髋关节高度约束
    eq1 = (ell_1 + ell_2) * np.cos(psi_hip_flex_fore) + ell_0 * np.cos(psi_hip_flex_fore + psi_ankle_fore) - c_2 * np.sin(phi_foot_fore) - \
          (ell_1 * np.cos(psi_hip_flex_rear) + ell_2 * np.cos(psi_hip_flex_rear + psi_knee_rear) + ell_0 * np.cos(phi_foot_rear) + c_1 * np.sin(phi_foot_rear))

    # 方程 (4)：步长约束
    eq2 = ell / 2 - c - (
        -(ell_1 + ell_2) * np.sin(psi_hip_flex_fore) - ell_0 * np.sin(psi_hip_flex_fore + psi_ankle_fore) - c_2 * np.cos(phi_foot_fore) +
        ell_1 * np.sin(psi_hip_flex_rear) + ell_2 * np.sin(psi_hip_flex_rear + psi_knee_rear) + ell_0 * np.sin(phi_foot_rear) - c_1 * np.cos(phi_foot_rear)
    )

    return [eq1, eq2]
    
# 绘制图像
plt.figure(figsize=(4.2,6),dpi=200)
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

# 后脚足底角度固定为 5°
phi_foot_rear = np.radians(5)  
# 参数范围
psi_knee_rear_vals = np.linspace(0, np.pi/3, 100)  # 后膝角度范围 [0, 30°]
psi_hip_flex_fore_vals = []
psi_hip_flex_rear_vals = []
# 解方程
for psi_knee_rear in psi_knee_rear_vals:
    # 初始猜测
    initial_guess = [-0.1, 0.1]  # [psi_hip_flex_fore, psi_hip_flex_rear]
    solution = fsolve(equations, initial_guess, args=(psi_knee_rear, phi_foot_rear))
    psi_hip_flex_fore_vals.append(solution[0])
    psi_hip_flex_rear_vals.append(solution[1])
#绘图
plt.plot(psi_knee_rear_vals, psi_hip_flex_fore_vals, label=r'$\psi_{\mathrm{hip,flex}}^{fore}$, $\varphi_{\mathrm{foot}}^{rear}=\pi/36$', color='C0', linewidth=2, linestyle='-')
plt.plot(psi_knee_rear_vals, psi_hip_flex_rear_vals, label=r'$\psi_{\mathrm{hip,flex}}^{rear}$, $\varphi_{\mathrm{foot}}^{rear}=\pi/36$', color='C0', linewidth=2, linestyle='--')


# 后脚足底角度固定为 10°
phi_foot_rear = np.radians(10)  
# 参数范围
psi_knee_rear_vals = np.linspace(0, np.pi/3, 100)  # 后膝角度范围 [0, 30°]
psi_hip_flex_fore_vals = []
psi_hip_flex_rear_vals = []
# 解方程
for psi_knee_rear in psi_knee_rear_vals:
    # 初始猜测
    initial_guess = [-0.1, 0.1]  # [psi_hip_flex_fore, psi_hip_flex_rear]
    solution = fsolve(equations, initial_guess, args=(psi_knee_rear, phi_foot_rear))
    psi_hip_flex_fore_vals.append(solution[0])
    psi_hip_flex_rear_vals.append(solution[1])
#绘图
plt.plot(psi_knee_rear_vals, psi_hip_flex_fore_vals, label=r'$\psi_{\mathrm{hip,flex}}^{fore}$, $\varphi_{\mathrm{foot}}^{rear}=\pi/18$', color='C1', linewidth=2, linestyle='-')
plt.plot(psi_knee_rear_vals, psi_hip_flex_rear_vals, label=r'$\psi_{\mathrm{hip,flex}}^{rear}$, $\varphi_{\mathrm{foot}}^{rear}=\pi/18$', color='C1', linewidth=2, linestyle='--')

# 后脚足底角度固定为 15°
phi_foot_rear = np.radians(15)  
# 参数范围
psi_knee_rear_vals = np.linspace(0, np.pi/3, 100)  # 后膝角度范围 [0, 30°]
psi_hip_flex_fore_vals = []
psi_hip_flex_rear_vals = []
# 解方程
for psi_knee_rear in psi_knee_rear_vals:
    # 初始猜测
    initial_guess = [-0.1, 0.1]  # [psi_hip_flex_fore, psi_hip_flex_rear]
    solution = fsolve(equations, initial_guess, args=(psi_knee_rear, phi_foot_rear))
    psi_hip_flex_fore_vals.append(solution[0])
    psi_hip_flex_rear_vals.append(solution[1])
#绘图
plt.plot(psi_knee_rear_vals, psi_hip_flex_fore_vals, label=r'$\psi_{\mathrm{hip,flex}}^{fore}$, $\varphi_{\mathrm{foot}}^{rear}=\pi/36$', color='C2', linewidth=2, linestyle='-')
plt.plot(psi_knee_rear_vals, psi_hip_flex_rear_vals, label=r'$\psi_{\mathrm{hip,flex}}^{rear}$, $\varphi_{\mathrm{foot}}^{rear}=\pi/36$', color='C2', linewidth=2, linestyle='--')


plt.xlabel(r'$\psi_{\mathrm{knee}}^{\mathrm{rear}}$ ')
#plt.ylabel(r'$\psi_{\mathrm{hip,flex}}$ ')
#plt.grid(True)
plt.legend(labelspacing=1,prop={'size':20},handlelength=1,borderpad=0.2)
plt.tight_layout()
plt.axis('equal')

plt.xlim(-np.pi/3,0)
plt.ylim(-np.pi,np.pi/6)

x_ticks = np.arange(0, np.pi/3+0.0001, np.pi/12)
x_tick_labels = ['0', r'$\frac{\pi}{12}$', r'$\frac{\pi}{6}$', r'$\frac{\pi}{4}$', r'$\frac{\pi}{3}$']
plt.xticks(x_ticks, x_tick_labels,fontsize=25)
y_ticks = np.arange(-np.pi/3, np.pi/6+0.0001, np.pi/12)
y_tick_labels = [r'$-\frac{\pi}{3}$',r'$-\frac{\pi}{4}$', r'$-\frac{\pi}{6}$', r'$-\frac{\pi}{12}$','0', r'$\frac{\pi}{12}$', r'$\frac{\pi}{6}$']
plt.yticks(y_ticks, y_tick_labels,fontsize=25)
plt.tick_params(pad=8)
# 保存图像

plt.show()